# v1.1 Use of Force / OIS methodology

This notebook documents the working v1.1 methodology for Seattle Police Department **Use of Force (UOF)** and **Officer-Involved Shooting (OIS)** metrics.

## Purpose

1. Clarify the schema and counting grain of:
   - `ppi5-g2bj` — general SPD Use of Force stream
   - `mg5r-efcm` — dedicated SPD Officer-Involved Shooting research stream
2. Define the production counting methodology for general UOF incidents and unique OIS events.
3. Reproduce the historical OIS reconciliation QA that motivated the event-grouping rule.
4. Prototype KPI cards for UOF and OIS.

## Working production decision

The dashboard should use **`ppi5-g2bj` as the operational stream for both UOF and OIS metrics** because it is the fresher stream.

The dedicated OIS dataset (`mg5r-efcm`) remains a **historical validation/reference dataset**, not the production freshness-limiting feed.

### Counting units

- **General UOF KPI:** unique `incident_num`
- **OIS KPI:** unique derived shooting events from UOF records classified as OIS
- UOF OIS force-incident counts are **not** interpreted directly as shooting-event counts.

### Derived OIS event rule

Historical QA showed that a single OIS event can produce multiple UOF rows, force-incident IDs, subject IDs, officer IDs, and occurrence timestamps.

The validated event grouping is therefore:

> **OIS event = unique combination of local calendar date + normalized SPD beat among UOF records classified as OIS.**

For OIS grouping only, unavailable / out-of-jurisdiction beat values such as `OOJ`, `99`, `-`, blank, and missing are normalized to a single `OUTSIDE_OR_UNKNOWN` geography state.

Across the historical overlap used during methodology development:

- 114 UOF OIS rows
- 97 unique UOF OIS force incidents
- 48 dedicated OIS events
- normalized `date + beat` produced 49 UOF-derived candidate events
- 47 of 48 comparison dates matched exactly
- the sole remaining mismatch was `2015-08-24`, where the general UOF stream contains one `Level 3 - OIS` record and the dedicated OIS research stream contains no corresponding event

That remaining disagreement is retained as a cross-source classification/inclusion discrepancy rather than silently removed to force agreement.

The dedicated `frb` field can sometimes link directly to the general UOF `uniqueid`, but exact linkage was incomplete and is therefore retained for QA only rather than used as the event key.


In [4]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
import requests
import plotly.graph_objects as go
from IPython.display import display

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dashboard.analysis_windows import (
    get_analysis_bounds,
    get_history_bounds,
    get_previous_period,
)

UOF_DATASET_ID = "ppi5-g2bj"
OIS_DATASET_ID = "mg5r-efcm"
SOCRATA_BASES = (
    "https://data.seattle.gov",
    "https://cos-data.seattle.gov",
)

session = requests.Session()
session.headers.update({
    "User-Agent": "Seattle-Area-Public-Safety-Dashboard/uof-ois-methodology"
})


def _get_json(path: str, *, params=None, timeout=60):
    last_error = None
    for base in SOCRATA_BASES:
        try:
            response = session.get(f"{base}{path}", params=params, timeout=timeout)
            response.raise_for_status()
            return response.json()
        except (requests.RequestException, ValueError) as exc:
            last_error = exc
    raise RuntimeError(f"Could not fetch Socrata resource {path}: {last_error}")


def fetch_schema(dataset_id: str) -> pd.DataFrame:
    metadata = _get_json(f"/api/views/{dataset_id}")
    return pd.DataFrame([
        {
            "name": column.get("name"),
            "field_name": column.get("fieldName"),
            "data_type": column.get("dataTypeName"),
            "description": column.get("description"),
        }
        for column in metadata.get("columns", [])
    ])


def fetch_all_rows(dataset_id: str, *, page_size: int = 50_000) -> pd.DataFrame:
    pages = []
    offset = 0
    while True:
        payload = _get_json(
            f"/resource/{dataset_id}.json",
            params={"$limit": page_size, "$offset": offset},
        )
        page = pd.DataFrame(payload)
        if page.empty:
            break
        pages.append(page)
        if len(page) < page_size:
            break
        offset += page_size
    return pd.concat(pages, ignore_index=True) if pages else pd.DataFrame()


## A. Schema and grain

The two datasets overlap conceptually but do **not** have the same row grain.

### General UOF stream — `ppi5-g2bj`

Important fields used here:

- `uniqueid` — force-record identifier
- `incident_num` — force-incident identifier
- `incident_type` — force classification, including OIS
- `occured_date_time` — occurrence timestamp
- `precinct`, `sector`, `beat` — SPD geography
- `officer_id`
- `subject_id`

A single shooting can produce multiple UOF rows and multiple `incident_num` values, so neither row count nor `incident_num.nunique()` should be interpreted as the number of shootings.

### Dedicated OIS stream — `mg5r-efcm`

Important fields used here:

- `go` — OIS case/event identifier
- `frb` — Force Review Board / UOF-related identifier where available
- `date_time` / `date` — event time/date
- event location fields
- officer characteristics
- subject characteristics
- fatality / weapon / review-related fields

The dedicated file can contain multiple rows for one OIS event. Event-level counting therefore uses unique `go`.

The next cells introspect the current live schemas rather than relying only on this written summary.


In [5]:
uof_schema = fetch_schema(UOF_DATASET_ID)
ois_schema = fetch_schema(OIS_DATASET_ID)

uof_schema["dataset"] = "UOF — ppi5-g2bj"
ois_schema["dataset"] = "OIS — mg5r-efcm"

schema_comparison = pd.concat([uof_schema, ois_schema], ignore_index=True)
display(schema_comparison[["dataset", "name", "field_name", "data_type", "description"]])


,dataset,name,field_name,data_type,description
0,UOF — ppi5-g2bj,ID,uniqueid,text,Composite key for the identification of use of...
1,UOF — ppi5-g2bj,Incident_Num,incident_num,text,Key identifying a force incident.
2,UOF — ppi5-g2bj,Incident_Type,incident_type,text,Use of force classification.
3,UOF — ppi5-g2bj,Occured_date_time,occured_date_time,calendar_date,Date and time that force occurred.
4,UOF — ppi5-g2bj,Precinct,precinct,text,Precinct where the force occurred.
5,UOF — ppi5-g2bj,Sector,sector,text,Sector where the force occurred.
6,UOF — ppi5-g2bj,Beat,beat,text,Beat where the force occurred.
7,UOF — ppi5-g2bj,Officer_ID,officer_id,text,Key identifying unique officers.
8,UOF — ppi5-g2bj,Subject_ID,subject_id,text,Key identifying unique subjects.
9,UOF — ppi5-g2bj,Subject_Race,subject_race,text,Race of the subject of the use of force


In [6]:
uof = fetch_all_rows(UOF_DATASET_ID)
ois = fetch_all_rows(OIS_DATASET_ID)

required_uof = {"uniqueid", "incident_num", "incident_type", "occured_date_time", "beat"}
required_ois = {"go", "date_time"}

missing_uof = required_uof - set(uof.columns)
missing_ois = required_ois - set(ois.columns)
if missing_uof:
    raise ValueError(f"UOF stream missing expected fields: {sorted(missing_uof)}")
if missing_ois:
    raise ValueError(f"OIS stream missing expected fields: {sorted(missing_ois)}")

uof["occured_date_time"] = pd.to_datetime(uof["occured_date_time"], errors="coerce")
ois["date"] = pd.to_datetime(ois["date_time"], errors="coerce")

grain_summary = pd.DataFrame([
    {
        "dataset": "UOF — ppi5-g2bj",
        "rows": len(uof),
        "unique_force_records": uof["uniqueid"].nunique(),
        "unique_force_incidents": uof["incident_num"].nunique(),
        "unique_ois_events": pd.NA,
        "earliest_date": uof["occured_date_time"].min(),
        "latest_date": uof["occured_date_time"].max(),
    },
    {
        "dataset": "OIS — mg5r-efcm",
        "rows": len(ois),
        "unique_force_records": pd.NA,
        "unique_force_incidents": pd.NA,
        "unique_ois_events": ois["go"].nunique(),
        "earliest_date": ois["date"].min(),
        "latest_date": ois["date"].max(),
    },
])

display(grain_summary)

print("UOF incident types containing OIS:")
display(
    uof.loc[
        uof["incident_type"].astype("string").str.contains(r"\bOIS\b", case=False, na=False, regex=True),
        "incident_type",
    ]
    .value_counts(dropna=False)
    .rename_axis("incident_type")
    .reset_index(name="rows")
)


C:\Users\benca\AppData\Local\Temp\ipykernel_29552\2134101383.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ois["date"] = pd.to_datetime(ois["date_time"], errors="coerce")


,dataset,rows,unique_force_records,unique_force_incidents,unique_ois_events,earliest_date,latest_date
0,UOF — ppi5-g2bj,19367,19367,18366,<NA>,2014-01-27 21:10:00,2026-08-28 21:25:00
1,OIS — mg5r-efcm,193,<NA>,<NA>,98,2005-03-21 18:28:00,2025-06-20 05:00:00


UOF incident types containing OIS:


,incident_type,rows
0,Level 3 - OIS,122


## B. Production OIS derivation

### Why UOF force incidents cannot be counted directly as shootings

Historical reconciliation showed that one dedicated OIS event can produce multiple UOF rows, multiple `incident_num` values, multiple officers, multiple subjects, and multiple occurrence timestamps.

Exact timestamp grouping and subject grouping both over-counted OIS events.

### Geography-based event grouping

The most stable reconstruction was **calendar date + normalized SPD beat**.

A key stress test was `2015-09-29`, when the dedicated dataset contained two distinct OIS events on the same date. The UOF stream also separated them geographically:

- West / KING / K1
- North / LINCOLN / L2

Therefore grouping by date alone would under-count, while date + beat preserves both events.

Another important case was `2024-04-17`. Two UOF records represented the same dedicated OIS event but had inconsistent out-of-jurisdiction geography (`-` versus `OOJ / 99`). Normalizing those unavailable/out-of-jurisdiction values to one common state correctly collapses them into a single event.

### Remaining discrepancy

`2015-08-24` remains a deliberate QA exception:

- the general UOF stream contains one `Level 3 - OIS` record,
- the dedicated OIS stream contains no corresponding event.

The operational stream retains the record because the source itself classifies it as OIS. It is not removed solely to force historical agreement.


In [7]:
OIS_LABEL_PATTERN = r"\bOIS\b"
UNKNOWN_OIS_GEOGRAPHY = {"", "-", "nan", "none", "<na>", "ooj", "99"}


def normalize_ois_geography(value) -> str:
    if pd.isna(value):
        return "OUTSIDE_OR_UNKNOWN"
    normalized = str(value).strip().lower()
    if normalized in UNKNOWN_OIS_GEOGRAPHY:
        return "OUTSIDE_OR_UNKNOWN"
    return normalized


def derive_uof_ois_events(uof_frame: pd.DataFrame) -> pd.DataFrame:
    work = uof_frame.loc[
        uof_frame["incident_type"].astype("string").str.contains(
            OIS_LABEL_PATTERN, case=False, na=False, regex=True
        )
    ].copy()

    work["event_date"] = pd.to_datetime(
        work["occured_date_time"], errors="coerce"
    ).dt.normalize()
    work["ois_beat_normalized"] = work["beat"].map(normalize_ois_geography)
    work = work.dropna(subset=["event_date"])

    events = (
        work.groupby(["event_date", "ois_beat_normalized"], as_index=False)
        .agg(
            uof_rows=("uniqueid", "nunique"),
            force_incidents=("incident_num", "nunique"),
            first_occurrence=("occured_date_time", "min"),
            last_occurrence=("occured_date_time", "max"),
        )
    )

    events["ois_event_key"] = (
        events["event_date"].dt.strftime("%Y-%m-%d")
        + "|"
        + events["ois_beat_normalized"]
    )
    return events


uof_ois_events = derive_uof_ois_events(uof)
display(uof_ois_events.sort_values(["event_date", "ois_beat_normalized"]))


,event_date,ois_beat_normalized,uof_rows,force_incidents,first_occurrence,last_occurrence,ois_event_key
0,2014-04-03,c3,1,1,2014-04-03 11:39:00,2014-04-03 11:39:00,2014-04-03|c3
1,2014-07-01,s1,1,1,2014-07-01 22:43:00,2014-07-01 22:43:00,2014-07-01|s1
2,2014-07-19,r3,1,1,2014-07-19 01:27:00,2014-07-19 01:27:00,2014-07-19|r3
3,2014-07-30,b3,2,2,2014-07-30 03:05:00,2014-07-30 03:05:00,2014-07-30|b3
4,2014-08-30,q3,2,2,2014-08-30 22:13:00,2014-08-30 22:16:00,2014-08-30|q3
5,2014-09-04,u2,1,1,2014-09-04 16:52:00,2014-09-04 16:52:00,2014-09-04|u2
6,2014-12-31,s2,15,3,2014-12-31 21:55:00,2014-12-31 21:55:00,2014-12-31|s2
7,2015-07-17,u1,1,1,2015-07-17 04:39:00,2015-07-17 04:39:00,2015-07-17|u1
8,2015-08-24,q2,1,1,2015-08-24 02:45:00,2015-08-24 02:45:00,2015-08-24|q2
9,2015-09-29,k1,1,1,2015-09-29 03:00:00,2015-09-29 03:00:00,2015-09-29|k1


## C. Historical reconciliation QA

This section validates the UOF-derived event count against the dedicated OIS research stream over their common observable period.

The goal is **not** to force the totals to match by deleting records. The goal is to confirm that the event-grouping rule reproduces the dedicated event structure closely enough to justify using the fresher UOF stream operationally.

Expected historical result from methodology development:

- overlap begins `2014-04-03`
- dedicated OIS overlap ends `2025-06-20`
- UOF-derived normalized date + beat events: **49**
- dedicated OIS events: **48**
- matching dates: **47 / 48**
- sole count mismatch: `2015-08-24`


In [8]:
uof_ois_rows = uof.loc[
    uof["incident_type"].astype("string").str.contains(
        OIS_LABEL_PATTERN, case=False, na=False, regex=True
    )
].copy()

uof_ois_rows["event_date"] = uof_ois_rows["occured_date_time"].dt.normalize()
ois["event_date"] = ois["date"].dt.normalize()

overlap_start = max(uof_ois_rows["event_date"].min(), ois["event_date"].min())
overlap_end = min(uof_ois_rows["event_date"].max(), ois["event_date"].max())

uof_events_overlap = uof_ois_events.loc[
    uof_ois_events["event_date"].between(overlap_start, overlap_end)
].copy()
ois_overlap = ois.loc[ois["event_date"].between(overlap_start, overlap_end)].copy()

uof_counts_by_date = (
    uof_events_overlap.groupby("event_date", as_index=False)
    .agg(candidate_ois_events=("ois_event_key", "nunique"))
)

dedicated_counts_by_date = (
    ois_overlap.groupby("event_date", as_index=False)
    .agg(dedicated_ois_events=("go", "nunique"))
)

ois_reconciliation = (
    dedicated_counts_by_date.merge(uof_counts_by_date, on="event_date", how="outer")
    .fillna(0)
)

ois_reconciliation[["dedicated_ois_events", "candidate_ois_events"]] = (
    ois_reconciliation[["dedicated_ois_events", "candidate_ois_events"]].astype(int)
)
ois_reconciliation["counts_match"] = (
    ois_reconciliation["dedicated_ois_events"]
    == ois_reconciliation["candidate_ois_events"]
)

print("Overlap window:", overlap_start.date(), "to", overlap_end.date())
print(
    "UOF OIS rows in overlap:",
    len(uof_ois_rows.loc[uof_ois_rows["event_date"].between(overlap_start, overlap_end)]),
)
print("UOF-derived OIS events:", int(ois_reconciliation["candidate_ois_events"].sum()))
print("Dedicated OIS events:", int(ois_reconciliation["dedicated_ois_events"].sum()))
print("Matching dates:", int(ois_reconciliation["counts_match"].sum()), "/", len(ois_reconciliation))

display(ois_reconciliation.loc[~ois_reconciliation["counts_match"]])


Overlap window: 2014-04-03 to 2025-06-20
UOF OIS rows in overlap: 114
UOF-derived OIS events: 49
Dedicated OIS events: 48
Matching dates: 47 / 48


,event_date,dedicated_ois_events,candidate_ois_events,counts_match
8,2015-08-24,0,1,False


### FRB / UOF record linkage QA

Some dedicated OIS `frb` identifiers correspond directly to general UOF `uniqueid` values, but historical linkage is incomplete.

During methodology development:

- UOF OIS records in overlap: 114
- dedicated FRB identifiers: 36
- exact linked records: 28
- UOF OIS records without an exact FRB link: 86
- dedicated FRB identifiers without an exact UOF link: 8

Therefore direct `frb ↔ uniqueid` matching is useful for spot-checking but is **not** used as the production event key.


In [9]:
def normalize_force_record_id(value):
    if pd.isna(value):
        return pd.NA
    return str(value).strip().upper().replace(" ", "")


if "frb" in ois_overlap.columns:
    uof_overlap_rows = uof_ois_rows.loc[
        uof_ois_rows["event_date"].between(overlap_start, overlap_end)
    ].copy()

    uof_overlap_rows["uof_id_normalized"] = (
        uof_overlap_rows["uniqueid"].map(normalize_force_record_id)
    )
    ois_overlap["frb_normalized"] = ois_overlap["frb"].map(normalize_force_record_id)

    uof_ids = set(uof_overlap_rows["uof_id_normalized"].dropna())
    frb_ids = set(ois_overlap["frb_normalized"].dropna())

    print("UOF OIS records:", len(uof_ids))
    print("Dedicated FRB identifiers:", len(frb_ids))
    print("Exact linked records:", len(uof_ids & frb_ids))
    print("UOF records not directly linked:", len(uof_ids - frb_ids))
    print("Dedicated FRB IDs not directly linked:", len(frb_ids - uof_ids))
else:
    print("Dedicated OIS schema does not currently expose an 'frb' field.")


UOF OIS records: 114
Dedicated FRB identifiers: 36
Exact linked records: 28
UOF records not directly linked: 86
Dedicated FRB IDs not directly linked: 8


## D. KPI proposal

### Primary KPI 1 — Use-of-force incidents

**Definition:** count unique `incident_num` values from the general UOF stream during the selected analysis period.

**Display:**

- current-period unique UOF incidents
- raw change from the immediately preceding equal-length period
- optional percentage change
- subtitle should state **unique force incidents**, not force records

### Primary KPI 2 — Officer-involved shootings

**Definition:** count unique derived `ois_event_key` values from UOF records classified as OIS during the selected analysis period.

**Display:**

- current-period unique OIS events
- raw change from the immediately preceding equal-length period
- optional percentage change
- subtitle should state **derived unique OIS events**
- an info/disclaimer control should explain that event grouping is derived from the operational UOF stream and historically validated against the dedicated OIS research dataset

### Shared time behavior

Both cards should use the dashboard's selected analysis period and the same immediately preceding equal-length comparison period.

Because both production KPIs are derived from `ppi5-g2bj`, they can share the same freshness boundary instead of allowing the slower dedicated OIS research file to limit dashboard recency.

### Metrics not recommended as primary cards

Do not use these as the headline OIS KPI:

- UOF rows classified as OIS
- unique OIS `incident_num` force incidents
- officer count
- subject count

Those may be useful in drill-downs or QA, but they do not represent the number of shooting events.


In [10]:
def safe_pct_change(current, previous):
    if previous is None or pd.isna(previous) or previous == 0:
        return pd.NA
    return (current - previous) / previous * 100


def selected_period_counts(uof_frame: pd.DataFrame, ois_events: pd.DataFrame):
    uof_dates = pd.to_datetime(
        uof_frame["occured_date_time"], errors="coerce"
    ).dt.normalize()

    history_bounds = get_history_bounds(uof_dates)
    current_start, current_end = get_analysis_bounds(history_bounds[1])
    previous = get_previous_period(
        current_start,
        current_end,
        history_bounds=history_bounds,
    )
    if previous is None:
        raise ValueError("Insufficient UOF history for previous-period comparison.")
    previous_start, previous_end = previous

    current_uof_mask = uof_dates.between(current_start, current_end)
    previous_uof_mask = uof_dates.between(previous_start, previous_end)
    current_ois_mask = ois_events["event_date"].between(current_start, current_end)
    previous_ois_mask = ois_events["event_date"].between(previous_start, previous_end)

    current_uof = uof_frame.loc[current_uof_mask, "incident_num"].nunique()
    previous_uof = uof_frame.loc[previous_uof_mask, "incident_num"].nunique()
    current_ois = ois_events.loc[current_ois_mask, "ois_event_key"].nunique()
    previous_ois = ois_events.loc[previous_ois_mask, "ois_event_key"].nunique()

    return pd.DataFrame([
        {
            "metric": "Use-of-force incidents",
            "current": current_uof,
            "previous": previous_uof,
            "raw_change": current_uof - previous_uof,
            "pct_change": safe_pct_change(current_uof, previous_uof),
            "current_start": current_start,
            "current_end": current_end,
        },
        {
            "metric": "Officer-involved shootings",
            "current": current_ois,
            "previous": previous_ois,
            "raw_change": current_ois - previous_ois,
            "pct_change": safe_pct_change(current_ois, previous_ois),
            "current_start": current_start,
            "current_end": current_end,
        },
    ])


kpi_table = selected_period_counts(uof, uof_ois_events)
display(kpi_table)


,metric,current,previous,raw_change,pct_change,current_start,current_end
0,Use-of-force incidents,689,880,-191,-21.704545,2025-08-28,2026-08-28
1,Officer-involved shootings,3,3,0,0.000000,2025-08-28,2026-08-28


In [11]:
def make_kpi_figure(title: str, current: int, previous: int, *, subtitle: str) -> go.Figure:
    fig = go.Figure(
        go.Indicator(
            mode="number+delta",
            value=current,
            number={"valueformat": ",d"},
            delta={
                "reference": previous,
                "relative": False,
                "valueformat": ",d",
            },
            title={
                "text": (
                    f"<b>{title}</b><br>"
                    f"<span style='font-size:0.75em'>{subtitle}</span>"
                )
            },
        )
    )
    fig.update_layout(height=250, margin=dict(l=30, r=30, t=60, b=30))
    return fig


uof_kpi = kpi_table.loc[kpi_table["metric"] == "Use-of-force incidents"].iloc[0]
ois_kpi = kpi_table.loc[kpi_table["metric"] == "Officer-involved shootings"].iloc[0]

make_kpi_figure(
    "Use-of-force incidents",
    int(uof_kpi["current"]),
    int(uof_kpi["previous"]),
    subtitle=(
        "Unique force incidents · "
        f"{uof_kpi['current_start'].date()} to {uof_kpi['current_end'].date()}"
    ),
).show()

make_kpi_figure(
    "Officer-involved shootings",
    int(ois_kpi["current"]),
    int(ois_kpi["previous"]),
    subtitle=(
        "Derived unique OIS events · "
        f"{ois_kpi['current_start'].date()} to {ois_kpi['current_end'].date()}"
    ),
).show()


## E. Production carry-forward

Before dashboard deployment, the production implementation should include automated QA for:

1. schema drift in both source datasets;
2. unexpected new OIS `incident_type` labels;
3. new out-of-jurisdiction / missing beat encodings;
4. duplicate derived `ois_event_key` behavior;
5. comparison of newly reviewed dedicated OIS events against previously derived UOF events;
6. the known `2015-08-24` historical cross-source discrepancy.

The dedicated OIS research stream should continue to be checked periodically. It remains valuable as an external validation set even though it should not determine the live dashboard's freshness.

The methodology should be revisited if future dedicated OIS releases produce systematic disagreement with the UOF-derived `date + normalized beat` event key.
